### Vector Stores
Vector Stores are databases optimized to find similar vectors. They enable semantic search. 

### `InMemoryVectorStore` simple to use

In [1]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embedding=embeddings)

docs = [
    Document(page_content="Python is a programming language"),
    Document(page_content="Machine learning uses algorithms"),
    Document(page_content="The sun is a star")
]

vector_store.add_documents(docs)
print(f"Added {len(docs)} documents\n")

results = vector_store.similarity_search("What is Python?",k=2)
print("Search results")

for i,doc in enumerate(results,1):
    print(f"{i}. {doc.page_content}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Added 3 documents

Search results
1. Python is a programming language
2. Machine learning uses algorithms


### `FAISS` 
This is fast and production ready

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

pdf_path = "attention.pdf"

if Path(pdf_path).exists():
    print("Loading pdf document")
    loader = PyPDFLoader(pdf_path)

    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                              chunk_overlap=200)
    
    chunks = splitter.split_documents(docs)
    print(f"Split into {len(chunks)} chunks")

    print("Creating the embeddings this may take a minute")
    
    vector_store = FAISS.from_documents(chunks,embeddings)
    vector_store.save_local("./faiss_index_notebooks")
    print("FAISS index saved")

    query = "What is attention mechanism"
    results = vector_store.similarity_search(query,k=3)
    print(f"Query: {query}\n")

    for i,doc in enumerate(results,1):
        print(f"Result: {i}")
        print(f" {doc.page_content[:150]}...\n")

else:
    print(f"PDF not found: {pdf_path}")

C:\Users\rsurs\AppData\Local\Temp\ipykernel_7948\1323790668.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading pdf document
Split into 52 chunks
Creating the embeddings this may take a minute
FAISS index saved
Query: What is attention mechanism

Result: 1
 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and ...

Result: 2
 Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
re...

Result: 3
 Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
...



In [4]:
if Path("./faiss_index_notebook").exists():
    loaded_vectorstore = FAISS.load_local("./faiss_index_notebook",
                                          embeddings,
                                          allow_dangerous_deserialization=True)
    print("Load existing FAISS index")
    result = loaded_vectorstore.similarity_search("transformers",k=2)
    print(f"\nFound {len(results)} results")
else:
    print("No existing index found")

No existing index found


### Chroma (persistent and easy)

In [5]:
from langchain_chroma import Chroma 

chroma = Chroma(collection_name="my_collection",
                embedding_function=embeddings,
                persist_directory="./chroma_db_notebook")


sample_docs = [
    Document(
        page_content="RAG combines retrieval and generation",
        metadata={"topic": "rag", "difficulty": "intermediate"}
    ),
    Document(
        page_content="LangChain simplifies LLM applications",
        metadata={"topic": "langchain", "difficulty": "beginner"}
    )
]

chroma.add_documents(sample_docs)
print("Added documents to Chroma\n")

results = chroma.similarity_search("Tell me about rag",
                                   k=2,
                                   filter={"topic":"rag"})
print("Filtered search results:")
for doc in results:
    print(f"  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")

Added documents to Chroma

Filtered search results:
  RAG combines retrieval and generation
  Metadata: {'topic': 'rag', 'difficulty': 'intermediate'}
